# HW2: verify the 180,000-step model on one T4
Run each phase separately; do not use Run all. First select **CPU** and finish file checks. Then select **T4** for generation. The GPU phase saves results to Drive and releases the runtime. Export from Drive afterward using CPU.

The checkpoint was trained from random weights on the allowed training split. Its public score is 47.40% (455/960). All 1,919 programs preserve the official test.parquet question order, as confirmed by the user. Every program is an exact model decode. The timer includes tokenization, sorting, generation and CSV writing. A fresh T4 measurement is still required.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU for file preparation'
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)

In [ ]:
import os, sys, json, hashlib, shutil, tarfile, subprocess
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Keep GPU off during file checks'
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
assert Path('/content/drive').is_mount()
prepared = root/'prepared/inference-180000-v1'
prepared.mkdir(parents=True, exist_ok=True)
work = Path('/content/hw2-inference-prepared')
work.mkdir(exist_ok=True)
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''): h.update(block)
    return h.hexdigest()
source = root/'prepared/continuation-60k-v1'
source_ready = json.loads((source/'ready.json').read_text())
assert source_ready['source_commit'] == 'ca26869455692cb570b94cb4a633be7701221f9e'
assert digest(source/'source.tar') == source_ready['source_sha256']
with tarfile.open(source/'source.tar') as z: z.extractall(work, filter='data')
os.chdir(work)
sys.path.insert(0, str(work))
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.checkpoint_io import drive_preflight
from homeworks.hw2.data import fetch_parquet
drive_preflight(root)
parent = root/'runs/continue-60k-180k-b128-v1/step-180000'
manifest = json.loads((parent/'checkpoint.complete.json').read_text())
assert manifest['status'] == 'checkpoint_verified'
assert manifest['files']['model.safetensors'] == '0399a924eea6ef872aa0b403b77fc5e33d24909cc7690536880bc00b254362c7'
model_dir = work/'model'; model_dir.mkdir(exist_ok=True)
for name, expected in manifest['files'].items():
    assert Path(name).name == name
    assert digest(parent/name) == expected, name
    if name != 'training_state.pt':
        shutil.copyfile(parent/name, model_dir/name)
        assert digest(model_dir/name) == expected
shutil.copyfile(parent/'checkpoint.complete.json', model_dir/'checkpoint.complete.json')
test = fetch_parquet('data/test/test.parquet', work/'cache')
import pyarrow.parquet as pq
questions = pq.read_table(test,columns=['question'])['question'].to_pylist()
question_hash = hashlib.sha256(json.dumps(questions,ensure_ascii=False).encode()).hexdigest()
assert len(questions) == 1919
assert question_hash == '214746f8d4bc918b6fa1fde8ca3185a518a60637f239a2c45efa9a8143f612d3'
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True)
assert sum(p.numel() for p in model.parameters()) == 9967360
del model
bundle = Path('/content/inference-inputs.tar')
with tarfile.open(bundle,'w') as z:
    for folder in ['homeworks','model','cache']:
        z.add(work/folder, arcname=folder)
    z.add(work/'source_commit.txt',arcname='source_commit.txt')
target = prepared/'inputs.tar'
shutil.copyfile(bundle,target)
assert digest(target) == digest(bundle)
ready = {'source_commit':source_ready['source_commit'],'bundle_sha256':digest(target),
         'model_sha256':manifest['files']['model.safetensors'],'questions_sha256':question_hash,
         'rows':1919,'cpu_load_verified':True}
(prepared/'ready.json').write_text(json.dumps(ready,indent=2))
assert json.loads((prepared/'ready.json').read_text()) == ready
print('CPU_INFERENCE_READY',ready)

## T4 phase
Switch to a T4 only after CPU_INFERENCE_READY. The next cell restores the small verified bundle, generates predictions, saves them to Drive and releases the GPU in a finally block. It does not download the training dataset.


In [ ]:
from google.colab import runtime
try:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=300000)
    from pathlib import Path
    import os, sys, json, hashlib, shutil, tarfile, subprocess
    root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
    prepared = root/'prepared/inference-180000-v1'
    ready = json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_load_verified'] and ready['rows'] == 1919
    work = Path('/content/hw2-inference-t4'); work.mkdir(exist_ok=True)
    bundle = Path('/content/inference-inputs.tar')
    shutil.copyfile(prepared/'inputs.tar',bundle)
    assert hashlib.sha256(bundle.read_bytes()).hexdigest() == ready['bundle_sha256']
    with tarfile.open(bundle) as z: z.extractall(work,filter='data')
    os.chdir(work); sys.path.insert(0,str(work))
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
    from homeworks.hw2.checkpoint_io import sha256, drive_preflight, package_jsonl
    drive_preflight(root)
    model_dir = work/'model'
    assert sha256(model_dir/'model.safetensors') == ready['model_sha256']
    print('T4_INPUTS_RESTORED')
    import torch, csv, datetime
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import runtime
    from homeworks.hw2.inference import generate_predictions
    from transformers import PreTrainedTokenizerFast
    output = root/'inference'/('step-180000-t4-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    output.mkdir(parents=True,exist_ok=False)
    assert torch.cuda.is_available(), 'Select a T4 GPU runtime'
    gpu = torch.cuda.get_device_name(0)
    assert torch.cuda.device_count() == 1 and 'T4' in gpu, gpu
    local = work/'results'; local.mkdir(exist_ok=True)
    count, seconds = generate_predictions(model_dir,local/'predictions.csv',work/'cache',
                                         batch_size=64,max_new_tokens=192,precision='fp32')
    assert count == 1919
    provenance = json.loads((local/'predictions.provenance.json').read_text())
    assert provenance['model_sha256'] == ready['model_sha256']
    assert provenance['questions_sha256'] == ready['questions_sha256']
    assert provenance['handwritten_repairs'] == 0
    with (local/'predictions.csv').open(newline='') as f:
        programs = [r['code'] for r in csv.DictReader(f)]
    tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir)
    assert programs == tokenizer.batch_decode(provenance['selected_token_ids'],skip_special_tokens=True,clean_up_tokenization_spaces=False)
    jsonl_hash = package_jsonl(local/'predictions.csv',local/'predictions.jsonl')
    evidence = {'count':count,'generation_seconds':seconds,'gpu':gpu,'t4_within_180_seconds':seconds<=180,
                'jsonl_sha256':jsonl_hash,'checkpoint':'continue-60k-180k-b128-v1/step-180000',
                'question_order':'official data/test/test.parquet','source_commit':ready['source_commit']}
    (local/'timing.json').write_text(json.dumps(evidence,indent=2))
    for name in ['predictions.csv','predictions.jsonl','predictions.provenance.json','timing.json']:
        shutil.copyfile(local/name,output/name)
        assert sha256(local/name) == sha256(output/name)
    # Build and export only from verified Drive copies.
    with ZipFile(output/'inference-results.zip','w',ZIP_DEFLATED) as z:
        for name in ['predictions.csv','predictions.jsonl','predictions.provenance.json','timing.json']:
            z.write(output/name,arcname=name)
    (output/'results.complete.json').write_text(json.dumps(evidence,indent=2))
    (prepared/'last-result.json').write_text(json.dumps({'path':str(output)}))
    print('SAVED_TO_DRIVE',str(output),evidence)
    assert seconds <= 180, 'Outputs saved, but T4 timing requirement failed'
finally:
    try:
        runtime.unassign()
    except Exception as error:
        print('Release failed: disconnect and delete runtime manually.',type(error).__name__)

## Export with CPU
Verify Runtime > Manage sessions has no active GPU session. Select CPU and run this cell to download the saved Drive archive. Submit its predictions.jsonl to the course bot and retain the returned score with its SHA256. A T4 run can produce different exact outputs from Blackwell, so it needs its own bot receipt.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU before exporting'
from google.colab import drive, files
drive.mount('/content/drive', timeout_ms=300000)
import json
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
output = Path(json.loads((root/'prepared/inference-180000-v1/last-result.json').read_text())['path'])
assert output.is_relative_to(root/'inference')
print(json.loads((output/'results.complete.json').read_text()))
files.download(str(output/'inference-results.zip'))